In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("AAFAQ_Final_Cleaned.csv")
df.head()

,QuestionText,Category,Answer,QuestionText_Classification
0,ايهما افضل الدراسه في السابق ام في الوقت الحالي,التعليم,الدراسه في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايه فضل درس في سبق ام في وقت الحالي
1,اليس القطن عماد الثروه في مصر,الاقتصاد والعمل,القطن يعتبر من اهم المنتجات الزراعيه في مصر وي...,الس قطن عمد ثره في مصر
2,اتصعد الشمس من الشرق,التعليم,الشمس تصعد من الشرق,صعد شمس من شرق
3,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تعرف بانها كاينات حيه دقيقه,عرف كتر بان كين حيه دقق
4,ايتكون الهوا اساسا من النيتروجين,التعليم,الهوا يتكون اساسا من النيتروجين,ايت هوا سسا من ترج


In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00


In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

In [ ]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True)

In [ ]:
X_train=Dataset.from_pandas(df_train.reset_index(drop=True))
X_val=Dataset.from_pandas(df_val.reset_index(drop=True))
X_test=Dataset.from_pandas(df_test.reset_index(drop=True))

# **T5:**

In [ ]:
model = "google/mt5-small"
tokenizer_T5 = AutoTokenizer.from_pretrained(model)
model_T5 = AutoModelForSeq2SeqLM.from_pretrained(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess_data_T5(examples):
    questions = df['QuestionText'].tolist()
    answers = df['Answer'].tolist()
    inputs_T5 = tokenizer_T5(questions, text_target=answers, max_length=128, truncation=True, padding=True)
    labels_T5 = tokenizer_T5(text_target=answers, max_length=128, truncation=True, padding=True)
    inputs_T5["labels"] = labels_T5["input_ids"]
    return inputs_T5

In [ ]:
X_T5_train = X_train.map(preprocess_data_T5, batched=True, remove_columns=X_train.column_names)
X_T5_val = X_val.map(preprocess_data_T5, batched=True, remove_columns=X_val.column_names)
X_T5_test = X_test.map(preprocess_data_T5, batched=True, remove_columns=X_test.column_names)

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_T5,
    model=model_T5
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_qa_model",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch"
)

In [ ]:
trainer_T5 = Seq2SeqTrainer(
    model=model_T5,
    args=training_args,
    train_dataset=X_T5_train,
    eval_dataset=X_T5_val,
    data_collator=data_collator
)

trainer_T5.train()

Epoch,Training Loss,Validation Loss
1,1.037074,0.776233
2,0.908640,0.703424


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12524, training_loss=2.168198213756979, metrics={'train_runtime': 2292.1927, 'train_samples_per_second': 21.852, 'train_steps_per_second': 5.464, 'total_flos': 2638161470361600.0, 'train_loss': 2.168198213756979, 'epoch': 2.0})

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


In [ ]:
references = df_test['Answer'].tolist()
predictions = []

for i in range(len(df_test)):
    question_text = df_test.iloc[i]['QuestionText']
    input_enc = tokenizer_T5(
        question_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(model_T5.device)

    outputs = model_T5.generate(
        **input_enc,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    pred_text = tokenizer_T5.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred_text)

references = [r.lower().strip() for r in references]
predictions = [p.lower().strip() for p in predictions]

references_tok = [[r.split()] for r in references]
predictions_tok = [p.split() for p in predictions]

smooth = SmoothingFunction().method1
bleu_score = corpus_bleu(references_tok, predictions_tok, smoothing_function=smooth)

print("BLEU Score (T5 QA):", bleu_score)

BLEU Score (T5 QA): 0.14727994757952811


In [ ]:
model_T5.save_pretrained("/content/drive/MyDrive/best_qa_model")
tokenizer_T5.save_pretrained("/content/drive/MyDrive/best_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/best_qa_model/tokenizer_config.json',
 '/content/drive/MyDrive/best_qa_model/tokenizer.json')

# **GPT:**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
model = "aubmindlab/aragpt2-base"
tokenizer_GPT = AutoTokenizer.from_pretrained(model)
model_GPT = AutoModelForCausalLM.from_pretrained(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tokenizer_GPT.add_special_tokens({'pad_token': '[PAD]'})
model_GPT.resize_token_embeddings(len(tokenizer_GPT))

Embedding(64001, 768)

In [ ]:
def preprocess_data_GPT(examples):
    questions = df['QuestionText'].tolist()
    answers = df['Answer'].tolist()
    inputs_GPT = tokenizer_GPT(questions, text_target=answers, max_length=128, truncation=True, padding="max_length")
    labels_GPT = tokenizer_GPT(text_target=answers, max_length=128, truncation=True, padding="max_length")
    inputs_GPT["labels"] = labels_GPT["input_ids"]
    return inputs_GPT
tokenizer_GPT.pad_token = tokenizer_GPT.eos_token


In [ ]:
X_GPT_train = X_train.map(preprocess_data_GPT, batched=True, remove_columns=X_train.column_names)
X_GPT_val = X_val.map(preprocess_data_GPT, batched=True, remove_columns=X_val.column_names)
X_GPT_test = X_test.map(preprocess_data_GPT, batched=True, remove_columns=X_test.column_names)

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_GPT,
    model=model_GPT
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./gpt_qa_model",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch"

)

In [ ]:
trainer_GPT = Seq2SeqTrainer(
    model=model_GPT,
    args=training_args,
    train_dataset=X_GPT_train,
    eval_dataset=X_GPT_val,
    data_collator=data_collator
)

trainer_GPT.train()

Epoch,Training Loss,Validation Loss
1,1.002956,0.858983
2,0.842810,0.705296


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12524, training_loss=1.0264501984376033, metrics={'train_runtime': 1123.3877, 'train_samples_per_second': 44.588, 'train_steps_per_second': 11.148, 'total_flos': 3272029470720000.0, 'train_loss': 1.0264501984376033, 'epoch': 2.0})

In [ ]:
references = df_test['Answer'].tolist()
predictions = []

for i in range(len(df_test)):
    question_text = df_test.iloc[i]['QuestionText']

    input_enc = tokenizer_GPT(
        question_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(model_GPT.device)

    outputs = model_GPT.generate(
        **input_enc,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    pred_text = tokenizer_GPT.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred_text)

references = [r.lower().strip() for r in references]
predictions = [p.lower().strip() for p in predictions]

references_tok = [[r.split()] for r in references]
predictions_tok = [p.split() for p in predictions]

smooth = SmoothingFunction().method1
bleu_score = corpus_bleu(references_tok, predictions_tok, smoothing_function=smooth)

print("BLEU Score (GPT QA):", bleu_score)

[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[tra

BLEU Score (GPT QA): 0.02852122872704796


# **QWEN**

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.0 MB/s eta 0:00:00


In [ ]:
import torch
import evaluate

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

In [ ]:
model = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_QWEN = AutoTokenizer.from_pretrained(model)
model_QWEN = AutoModelForCausalLM.from_pretrained(model,device_map="auto")
tokenizer_QWEN.pad_token = tokenizer_QWEN.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def preprocess_data_QWEN(examples):
    questions = df['QuestionText'].tolist()
    answers = df['Answer'].tolist()
    inputs_QWEN = tokenizer_QWEN(questions, text_target=answers, max_length=128, truncation=True, padding=True)
    labels_QWEN = tokenizer_QWEN(text_target=answers, max_length=128, truncation=True, padding=True)
    inputs_QWEN["labels"] = labels_QWEN["input_ids"]
    return inputs_QWEN

In [ ]:
X_QWEN_train = X_train.map(preprocess_data_QWEN, batched=True, remove_columns=X_train.column_names)
X_QWEN_val = X_val.map(preprocess_data_QWEN, batched=True, remove_columns=X_val.column_names)
X_QWEN_test = X_test.map(preprocess_data_QWEN, batched=True, remove_columns=X_test.column_names)

Map:   0%|          | 0/4007 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

In [ ]:
!pip install -U torchao

In [ ]:
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model_QWEN_lora = get_peft_model(model_QWEN, lora_config)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_QWEN,
    mlm=False
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_qa_lora",
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch"

)

In [ ]:
trainer_QWEN = Trainer(
    model=model_QWEN_lora,
    args=training_args,
    train_dataset=X_QWEN_train,
    eval_dataset=X_QWEN_val,
    data_collator=data_collator
)

trainer_QWEN.train()

Epoch,Training Loss,Validation Loss
1,1.671451,1.582708
2,1.405779,1.307236


TrainOutput(global_step=50090, training_loss=1.7624831195266977, metrics={'train_runtime': 5934.3672, 'train_samples_per_second': 8.441, 'train_steps_per_second': 8.441, 'total_flos': 5493984799311360.0, 'train_loss': 1.7624831195266977, 'epoch': 2.0})

In [ ]:
# Prepare references
references = df_test['Answer'].tolist()
predictions = []

# Generate predictions for the test set
for i in range(len(df_test)):
    question_text = df_test.iloc[i]['QuestionText']

    # Encode question for Qwen model
    input_enc = tokenizer_QWEN(
        question_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(model_QWEN.device)

    # Generate answer
    outputs = model_QWEN.generate(
        **input_enc,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    # Decode generated sequence
    pred_text = tokenizer_QWEN.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred_text)

# Lowercase and strip
references = [r.lower().strip() for r in references]
predictions = [p.lower().strip() for p in predictions]

# Tokenize for BLEU
references_tok = [[r.split()] for r in references]
predictions_tok = [p.split() for p in predictions]

# Compute BLEU with smoothing
smooth = SmoothingFunction().method1
bleu_score = corpus_bleu(references_tok, predictions_tok, smoothing_function=smooth)

print("BLEU Score (Qwen QA):", bleu_score)

BLEU Score (Qwen QA): 0.045302799507706666
